
# 8. MAKİNE ÖĞRENMESİ MODELLERİNİN EĞİTİLMESİ

**Proje:** Veri Madenciliği Teknikleri ile IoT/IIoT Ağlarında Siber Saldırı Tespiti ve Sınıflandırılması

**Veri Seti:** Edge-IIoTset

## 8.1. Logistic Regression Modelinin Eğitilmesi

**Amaç:** Ağ trafiği kayıtlarını normal trafik ve farklı siber saldırı türleri olmak üzere 15 sınıfa ayırabilen bir sınıflandırma modeli oluşturmak.

**Yöntem:** Feature Selection aşamasında seçilen 20 öznitelik kullanılarak Logistic Regression algoritması eğitilecektir.

- Eğitim verisi: 319.993 kayıt
- Hedef değişken: Attack_type
- Sınıf sayısı: 15
- Algoritma: Logistic Regression

Attack_label hedef değişkeni model girdilerine dahil edilmeyecektir.

**Model Seçim Gerekçesi:** Logistic Regression, diğer algoritmaların sonuçlarıyla karşılaştırılabilecek temel bir sınıflandırma modeli olarak seçilmiştir.

**Beklenen Çıktı:** Eğitilmiş Logistic Regression modeli, eğitim bilgileri ve kaydedilmiş model dosyası.

**Not:** Test verileri eğitim sırasında kullanılmayacak, model performansı daha sonraki değerlendirme aşamasında ölçülecektir.


In [1]:

import pandas as pd
import numpy as np
import joblib
from pathlib import Path
from sklearn.linear_model import LogisticRegression

# Proje klasörü
proje = Path(r"E:\Data mining\Network_Intrusion_Detection")

# Eğitim veri seti
dosya = (
    proje / "Calisma_Verileri"
    / "EdgeIIoT_Train_Secilmis.csv"
)

# Verileri okuma
df = pd.read_csv(dosya)

# Giriş öznitelikleri ve hedef değişken
X_train = df.drop(
    columns=["Attack_label", "Attack_type"]
)

y_train = df["Attack_type"]

# Eğitim verisinin doğrulanması
assert len(X_train) == 319993
assert X_train.shape[1] == 20
assert y_train.nunique() == 15
assert np.isfinite(X_train.to_numpy(dtype=float)).all()

# Logistic Regression modelini oluşturma
model = LogisticRegression(
    solver="lbfgs",
    max_iter=1000,
    random_state=42
)

# Modeli eğitme
model.fit(X_train, y_train)

# Model klasörünü oluşturma
model_klasoru = proje / "Modeller"
model_klasoru.mkdir(parents=True, exist_ok=True)

# Eğitilmiş modeli kaydetme
model_yolu = (
    model_klasoru
    / "08_01_Logistic_Regression.joblib"
)

joblib.dump(model, model_yolu)

# Eğitim sonuçları
print("LOGISTIC REGRESSION EĞİTİM SONUÇLARI")
print("-" * 45)

print("Eğitim kaydı:", len(X_train))
print("Giriş özniteliği:", X_train.shape[1])
print("Öğrenilen sınıf sayısı:", len(model.classes_))
print("İterasyon sayısı:", model.n_iter_)
print("Model dosyası:", model_yolu)

print("\nModel eğitimi tamamlandı.")


LOGISTIC REGRESSION EĞİTİM SONUÇLARI
---------------------------------------------
Eğitim kaydı: 319993
Giriş özniteliği: 20
Öğrenilen sınıf sayısı: 15
İterasyon sayısı: [288]
Model dosyası: E:\Data mining\Network_Intrusion_Detection\Modeller\08_01_Logistic_Regression.joblib

Model eğitimi tamamlandı.



### 8.1. Logistic Regression — Sonuç ve Yorum

319.993 eğitim kaydı ve seçilen 20 öznitelik kullanılarak Logistic Regression sınıflandırma modeli eğitilmiştir.

**Elde Edilen Sonuçlar:**

- Eğitim kaydı: 319.993
- Giriş özniteliği: 20
- Öğrenilen trafik sınıfı: 15
- İterasyon sayısı: 288

**Değerlendirme:**

Model, eğitim verileri üzerinde 288 iterasyon sonucunda eğitim işlemini tamamlamıştır.

Eğitilen model `08_01_Logistic_Regression.joblib` dosyasına kaydedilmiştir.

Test verileri eğitim aşamasında kullanılmamıştır.

**Sonuç:**

Logistic Regression modeli başarıyla eğitilmiştir. Modelin sınıflandırma başarısı henüz ölçülmemiştir.

Accuracy, Precision, Recall ve F1-score sonuçları model değerlendirme aşamasında hesaplanacaktır.



## 8.2. Decision Tree (Karar Ağacı) Modelinin Eğitilmesi

**Amaç:** Karar ağacı algoritması kullanılarak 15 trafik sınıfını ayırt edebilen bir sınıflandırma modeli oluşturmak.

**Yöntem:** Seçilen 20 öznitelik ve 319.993 eğitim kaydı kullanılacaktır.

Aşırı öğrenme riskini sınırlandırmak amacıyla başlangıç parametreleri olarak max_depth=15 ve min_samples_leaf=10 belirlenecektir.

**Model Seçim Gerekçesi:** Decision Tree, sınıflandırma kararlarını yorumlanabilir kurallarla oluşturabildiği için tercih edilmiştir.

**Beklenen Çıktı:** Eğitilmiş karar ağacı modeli, ağacın derinliği, düğüm sayısı ve kaydedilen model dosyası.

**Not:** Parametreler başlangıç değerleridir. Nihai performans değerlendirmesi ve gerekli parametre iyileştirmeleri eğitim verileri üzerinden yapılacaktır. Test verileri eğitimde kullanılmayacaktır.


In [2]:

import pandas as pd
import joblib
from pathlib import Path
from sklearn.tree import DecisionTreeClassifier

# Proje klasörü
proje = Path(r"E:\Data mining\Network_Intrusion_Detection")

# Seçilmiş eğitim veri seti
dosya = proje / "Calisma_Verileri" / "EdgeIIoT_Train_Secilmis.csv"

df = pd.read_csv(dosya)

# Giriş ve hedef değişkenleri
X_train = df.drop(columns=["Attack_label", "Attack_type"])
y_train = df["Attack_type"]

# Decision Tree modelini oluşturma
dt_model = DecisionTreeClassifier(
    max_depth=15,
    min_samples_leaf=10,
    random_state=42
)

# Modeli eğitme
dt_model.fit(X_train, y_train)

# Eğitilmiş modeli kaydetme
model_klasoru = proje / "Modeller"
model_klasoru.mkdir(parents=True, exist_ok=True)

model_yolu = model_klasoru / "08_02_Decision_Tree.joblib"

joblib.dump(dt_model, model_yolu)

# Sonuçlar
print("DECISION TREE EĞİTİM SONUÇLARI")
print("-" * 45)

print("Eğitim kaydı:", len(X_train))
print("Giriş özniteliği:", X_train.shape[1])
print("Öğrenilen sınıf:", len(dt_model.classes_))
print("Ağaç derinliği:", dt_model.get_depth())
print("Düğüm sayısı:", dt_model.tree_.node_count)
print("Model dosyası:", model_yolu)

assert len(X_train) == 319993
assert X_train.shape[1] == 20
assert len(dt_model.classes_) == 15

print("\nDecision Tree eğitimi tamamlandı.")


DECISION TREE EĞİTİM SONUÇLARI
---------------------------------------------
Eğitim kaydı: 319993
Giriş özniteliği: 20
Öğrenilen sınıf: 15
Ağaç derinliği: 15
Düğüm sayısı: 291
Model dosyası: E:\Data mining\Network_Intrusion_Detection\Modeller\08_02_Decision_Tree.joblib

Decision Tree eğitimi tamamlandı.



### 8.2. Decision Tree — Sonuç ve Yorum

319.993 eğitim kaydı ve seçilen 20 öznitelik kullanılarak Decision Tree sınıflandırma modeli eğitilmiştir.

**Elde Edilen Sonuçlar:**

- Eğitim kaydı: 319.993
- Giriş özniteliği: 20
- Trafik sınıfı: 15
- Ağaç derinliği: 15
- Toplam düğüm sayısı: 291

**Değerlendirme:**

Model, max_depth=15 ve min_samples_leaf=10 parametreleri kullanılarak eğitilmiştir.

Eğitim sonucunda 15 seviyelik derinliğe ve 291 düğüme sahip bir karar ağacı oluşturulmuştur.

Eğitilen model `08_02_Decision_Tree.joblib` dosyasına kaydedilmiştir.

**Sonuç:**

Decision Tree modeli başarıyla eğitilmiştir. Modelin sınıflandırma başarısı henüz ölçülmemiştir.

Performans sonuçları, diğer algoritmalarla birlikte model değerlendirme aşamasında karşılaştırılacaktır.



## 8.3. Random Forest (Rastgele Orman) Modelinin Eğitilmesi

**Amaç:** Birden fazla karar ağacını birleştirerek ağ trafiğini 15 sınıfa ayırabilen bir Random Forest modeli oluşturmak.

**Yöntem:** Seçilen 20 öznitelik ve 319.993 eğitim kaydı kullanılacaktır.

Başlangıç parametreleri:

- n_estimators=100
- max_depth=18
- min_samples_leaf=5
- random_state=42

**Model Seçim Gerekçesi:** Random Forest, birden fazla karar ağacının sonuçlarını birleştiren ve doğrusal olmayan ilişkileri öğrenebilen bir topluluk öğrenme algoritmasıdır.

**Beklenen Çıktı:** Eğitilmiş Random Forest modeli, oluşturulan ağaç sayısı, sınıf sayısı ve kaydedilen model dosyası.

**Not:** Parametreler başlangıç amacıyla seçilmiştir. Model performansı daha sonraki değerlendirme aşamasında incelenecektir. Test verileri eğitimde kullanılmayacaktır.


In [3]:

import pandas as pd
import numpy as np
import joblib
from pathlib import Path
from sklearn.ensemble import RandomForestClassifier

# Proje klasörü
proje = Path(r"E:\Data mining\Network_Intrusion_Detection")

# Seçilmiş eğitim veri seti
dosya = proje / "Calisma_Verileri" / "EdgeIIoT_Train_Secilmis.csv"

# Eğitim verilerini okuma
df = pd.read_csv(dosya)

X_train = df.drop(
    columns=["Attack_label", "Attack_type"]
)

y_train = df["Attack_type"]

# Veri kontrolleri
assert len(X_train) == 319993
assert X_train.shape[1] == 20
assert y_train.nunique() == 15
assert np.isfinite(X_train.to_numpy(dtype=float)).all()

# Random Forest modelini oluşturma
rf_model = RandomForestClassifier(
    n_estimators=100,
    max_depth=18,
    min_samples_leaf=5,
    max_features="sqrt",
    random_state=42,
    n_jobs=-1
)

# Modeli eğitme
rf_model.fit(X_train, y_train)

# Modeli kaydetme
model_klasoru = proje / "Modeller"
model_klasoru.mkdir(parents=True, exist_ok=True)

model_yolu = model_klasoru / "08_03_Random_Forest.joblib"

joblib.dump(rf_model, model_yolu, compress=3)

# Eğitim bilgilerini görüntüleme
print("RANDOM FOREST EĞİTİM SONUÇLARI")
print("-" * 45)

print("Eğitim kaydı:", len(X_train))
print("Giriş özniteliği:", X_train.shape[1])
print("Öğrenilen sınıf:", len(rf_model.classes_))
print("Oluşturulan ağaç:", len(rf_model.estimators_))
print("Model dosyası:", model_yolu)

print("\nRandom Forest eğitimi tamamlandı.")


RANDOM FOREST EĞİTİM SONUÇLARI
---------------------------------------------
Eğitim kaydı: 319993
Giriş özniteliği: 20
Öğrenilen sınıf: 15
Oluşturulan ağaç: 100
Model dosyası: E:\Data mining\Network_Intrusion_Detection\Modeller\08_03_Random_Forest.joblib

Random Forest eğitimi tamamlandı.



### 8.3. Random Forest — Sonuç ve Yorum

319.993 eğitim kaydı ve seçilen 20 öznitelik kullanılarak Random Forest sınıflandırma modeli eğitilmiştir.

**Elde Edilen Sonuçlar:**

- Eğitim kaydı: 319.993
- Giriş özniteliği: 20
- Trafik sınıfı: 15
- Oluşturulan karar ağacı: 100

**Değerlendirme:**

Random Forest algoritması, 100 karar ağacından oluşan bir topluluk modeli olarak eğitilmiştir.

Modelin aşırı öğrenme riskini sınırlandırmak amacıyla ağaç derinliği ve yaprak başına minimum örnek sayısı belirlenmiştir.

Eğitilen model `08_03_Random_Forest.joblib` dosyasına kaydedilmiştir.

**Sonuç:**

Random Forest modelinin eğitimi başarıyla tamamlanmıştır. Test performansı henüz ölçülmemiştir.

Modelin başarısı, diğer algoritmalarla birlikte Accuracy, Precision, Recall ve F1-score ölçütleri kullanılarak değerlendirilecektir.



## 8.4. XGBoost Modelinin Eğitilmesi

**Amaç:** XGBoost algoritması kullanılarak 15 trafik sınıfını ayırt edebilen bir sınıflandırma modeli oluşturmak.

**Yöntem:** Seçilmiş 20 öznitelik ve 319.993 eğitim kaydı kullanılacaktır.

Saldırı türlerini içeren Attack_type hedef değişkeni, LabelEncoder ile 0–14 aralığındaki sayısal sınıf etiketlerine dönüştürülecektir.

Başlangıç parametreleri:

- n_estimators = 60
- max_depth = 5
- learning_rate = 0.1
- tree_method = hist
- random_state = 42

**Model Seçim Gerekçesi:** XGBoost, ardışık karar ağaçları ve gradient boosting yaklaşımıyla doğrusal olmayan ilişkileri öğrenebildiği için seçilmiştir.

**Beklenen Çıktı:** Eğitilmiş XGBoost modeli, 15 sınıfın doğrulanması ve model ile etiket kodlayıcısının kaydedilmesi.

**Not:** Parametreler başlangıç denemesi içindir. Test verileri eğitimde kullanılmayacak, model performansı daha sonra değerlendirilecektir.


In [5]:
%pip install xgboost

   ---------------------------------------- 0.0/48.9 MB ? eta -:--:--
   ---------------------------------------- 0.3/48.9 MB ? eta -:--:--
   - -------------------------------------- 1.8/48.9 MB 8.0 MB/s eta 0:00:06
   ---- ----------------------------------- 5.0/48.9 MB 11.6 MB/s eta 0:00:04
   ------ --------------------------------- 8.4/48.9 MB 13.1 MB/s eta 0:00:04
   --------- ------------------------------ 12.1/48.9 MB 14.1 MB/s eta 0:00:03
   ------------ --------------------------- 15.7/48.9 MB 14.8 MB/s eta 0:00:03
   --------------- ------------------------ 18.6/48.9 MB 15.0 MB/s eta 0:00:03
   ----------------- ---------------------- 21.5/48.9 MB 14.4 MB/s eta 0:00:02
   -------------------- ------------------- 24.6/48.9 MB 14.5 MB/s eta 0:00:02
   ----------------------- ---------------- 28.3/48.9 MB 14.8 MB/s eta 0:00:02
   -------------------------- ------------- 32.0/48.9 MB 15.0 MB/s eta 0:00:02
   ---------------------------- ----------- 35.4/48.9 MB 15.1 MB/s eta 0:0

In [1]:
import xgboost
print("XGBoost versiyası:", xgboost.__version__)

XGBoost versiyası: 3.4.1


In [2]:

import pandas as pd
import numpy as np
import joblib
from pathlib import Path
from sklearn.preprocessing import LabelEncoder
from xgboost import XGBClassifier

# Proje ve eğitim dosyası
proje = Path(r"E:\Data mining\Network_Intrusion_Detection")
dosya = proje / "Calisma_Verileri" / "EdgeIIoT_Train_Secilmis.csv"

df = pd.read_csv(dosya)

# Giriş ve hedef değişkenleri
X_train = df.drop(columns=["Attack_label", "Attack_type"])
y_train = df["Attack_type"]

# Sınıf etiketlerini sayısallaştırma
etiket_kodlayici = LabelEncoder()
y_kodlu = etiket_kodlayici.fit_transform(y_train)

# Veri doğrulama
assert len(X_train) == 319993
assert X_train.shape[1] == 20
assert len(etiket_kodlayici.classes_) == 15
assert np.isfinite(X_train.to_numpy()).all()

# XGBoost modelini oluşturma
xgb_model = XGBClassifier(
    n_estimators=60,
    max_depth=5,
    learning_rate=0.1,
    subsample=0.8,
    colsample_bytree=0.8,
    objective="multi:softprob",
    eval_metric="mlogloss",
    tree_method="hist",
    random_state=42,
    n_jobs=4
)

# Model eğitimi
xgb_model.fit(X_train, y_kodlu)

# Model ve etiket kodlayıcısını kaydetme
model_klasoru = proje / "Modeller"
model_klasoru.mkdir(parents=True, exist_ok=True)

joblib.dump(
    xgb_model,
    model_klasoru / "08_04_XGBoost.joblib"
)

joblib.dump(
    etiket_kodlayici,
    model_klasoru / "08_04_XGBoost_LabelEncoder.joblib"
)

# Eğitim sonuçları
print("XGBOOST EĞİTİM SONUÇLARI")
print("-" * 45)

print("Eğitim kaydı:", len(X_train))
print("Giriş özniteliği:", X_train.shape[1])
print("Öğrenilen sınıf:", len(xgb_model.classes_))
print("Boosting turu:", xgb_model.n_estimators)
print("Model kaydedildi:", "08_04_XGBoost.joblib")
print("Etiket kodlayıcı kaydedildi.")

print("\nXGBoost eğitimi tamamlandı.")


XGBOOST EĞİTİM SONUÇLARI
---------------------------------------------
Eğitim kaydı: 319993
Giriş özniteliği: 20
Öğrenilen sınıf: 15
Boosting turu: 60
Model kaydedildi: 08_04_XGBoost.joblib
Etiket kodlayıcı kaydedildi.

XGBoost eğitimi tamamlandı.



### 8.4. XGBoost — Sonuç ve Yorum

319.993 eğitim kaydı ve seçilen 20 öznitelik kullanılarak XGBoost sınıflandırma modeli eğitilmiştir.

**Elde Edilen Sonuçlar:**

- Eğitim kaydı: 319.993
- Giriş özniteliği: 20
- Trafik sınıfı: 15
- Boosting turu: 60

**Değerlendirme:**

Attack_type hedef değişkeni LabelEncoder kullanılarak sayısal sınıf etiketlerine dönüştürülmüştür.

XGBoost algoritması, 60 boosting turu ile eğitim işlemini tamamlamıştır.

Eğitilen model `08_04_XGBoost.joblib`, etiket kodlayıcısı ise `08_04_XGBoost_LabelEncoder.joblib` dosyasına kaydedilmiştir.

**Sonuç:**

XGBoost modeli başarıyla eğitilmiştir. Böylece Logistic Regression, Decision Tree, Random Forest ve XGBoost modellerinin ilk eğitimleri tamamlanmıştır.

Modellerin sınıflandırma performansları henüz ölçülmemiştir. Karşılaştırma, ayrılmış test veri kümesi üzerinde gerçekleştirilecektir.
